# Entrenamiento YOLOv11 — COCO + 3 clases nuevas

Este notebook configura el modelo **yolo11n** para mantener las clases COCO y añadir 3 clases nuevas (`cone`, `goal`, `ladder`). Incluye validación y predicción sobre imágenes de `val/`.

In [23]:
from ultralytics import YOLO
import os
from IPython.display import Image, display
import cv2

data_yaml = 'data.yaml'

# Modelo personalizado
custom_model = YOLO("yolo11s.pt")
custom_model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=8,
)

New https://pypi.org/project/ultralytics/8.3.229 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.228  Python-3.10.19 torch-2.9.1+cpu CPU (Intel Core i7-10510U 1.80GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train9, nbs=64, nms=False, opset=None, optimize=False

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000025D9D6EA800>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

In [25]:
coco_model = YOLO("yolo11n.pt")   # detecta person + sports ball
custom_model = YOLO("runs/detect/train9/weights/best.pt")  # detecta cones, goal, ladder


In [26]:
def detect_all(img_path):
    # === 1. Detección COCO ===
    coco_results = coco_model(img_path)[0]

    # === 2. Detección personalizada ===
    custom_results = custom_model(img_path)[0]

    # === 3. Leemos la imagen ===
    img = cv2.imread(img_path)

    # === 4. Dibujar detecciones COCO ===
    for box in coco_results.boxes:
        x1, y1, x2, y2 = box.xyxy[0]
        cls = int(box.cls[0])
        name = coco_model.names[cls]
        conf = float(box.conf[0])

        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0,255,0), 2)
        cv2.putText(img, f"{name} - conf: {conf:.2f}", (int(x1), int(y1)-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    # === 5. Dibujar detecciones PERSONALIZADAS ===
    for box in custom_results.boxes:
        x1, y1, x2, y2 = box.xyxy[0]
        cls = int(box.cls[0])
        name = custom_model.names[cls]
        conf = float(box.conf[0])

        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (255,0,0), 2)
        cv2.putText(img, f"{name} {conf:.2f}", (int(x1), int(y1)-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

    return img


In [28]:
output = detect_all("Labelling de conos_escalera y arco/val/images/ca8b40ce-Screenshot_2025-11-16_211343.png")

cv2.imwrite("prediccion_final.jpg", output)
print("Predicción guardada en prediccion_final.jpg")


image 1/1 c:\Users\Daniel\source\repos\curso-computer-vision-platzi\5__Entrenamiento_Modelo_Personalizado\Labelling de conos_escalera y arco\val\images\ca8b40ce-Screenshot_2025-11-16_211343.png: 320x640 10 persons, 2 sports balls, 100.7ms
Speed: 2.8ms preprocess, 100.7ms inference, 2.7ms postprocess per image at shape (1, 3, 320, 640)

image 1/1 c:\Users\Daniel\source\repos\curso-computer-vision-platzi\5__Entrenamiento_Modelo_Personalizado\Labelling de conos_escalera y arco\val\images\ca8b40ce-Screenshot_2025-11-16_211343.png: 320x640 3 cones, 3 goals, 199.4ms
Speed: 2.4ms preprocess, 199.4ms inference, 2.3ms postprocess per image at shape (1, 3, 320, 640)
Predicción guardada en prediccion_final.jpg
